In [1]:
import optuna
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import StandardScaler

from adbench.run_new import RunPipeline
from adbench.myutils_new import Utils
from MSML_v9 import MSML


# -------------------------------------------------
# Optuna Objective (FULL ADBench protocol)
# -------------------------------------------------
def objective(trial):

    # ---------- Hyperparameter Search Space ----------
    k = trial.suggest_int("k", 10, 100)
    nbd_sample_count_threshold = trial.suggest_int(
        "nbd_sample_count_threshold", 5, 80
    )
    learning_rate = trial.suggest_float(
        "learning_rate", 0.05, 1.0, log=True
    )
    max_iters_shift = trial.suggest_int("max_iters_shift", 5, 20)
    shift_threshold = trial.suggest_float(
        "shift_threshold", 1e-5, 1e-2, log=True
    )
    anomalyThreshold = trial.suggest_float(
        "anomalyThreshold", 0.01, 0.3
    )

    # ---------- Customized MSML Wrapper ----------
    class OptunaMSML(MSML):
        def __init__(self, seed, model_name=None):
            super().__init__(
                seed=seed,
                k=k,
                nbd_sample_count_threshold=nbd_sample_count_threshold,
                learning_rate=learning_rate,
                max_iters_shift=max_iters_shift,
                shift_threshold=shift_threshold,
                anomalyThreshold=anomalyThreshold,
                scaler=StandardScaler()
            )

    # ---------- ADBench Pipeline (FULL SETTING) ----------
    pipeline = RunPipeline(
        suffix="Optuna_MSML_FULL",
        parallel="unsupervise",
        realistic_synthetic_mode="dependency",
        noise_type=None
    )

    # ---------- Run FULL ADBench ----------
    pipeline.run(clf=OptunaMSML)

    # ---------- Load AUCROC Results ----------
    result_path = os.path.join(
    "adbench",
    "result",
    f"AUCROC_{pipeline.suffix}.csv"
    )

    df_aucroc = pd.read_csv(result_path, index_col=0)

    # ---------- Compute Mean AUCROC ----------
    mean_aucroc = np.nanmean(df_aucroc.values)

    return float(mean_aucroc)


# -------------------------------------------------
# Optuna Callback (print after each trial)
# -------------------------------------------------
def print_trial_result(study, trial):
    print("\n================ Trial Finished ================")
    print(f"Trial number : {trial.number}")
    print(f"AUCROC       : {trial.value}")
    print("Hyperparameters:")
    for k, v in trial.params.items():
        print(f"  {k}: {v}")
    print("================================================\n")


# -------------------------------------------------
# Main
# -------------------------------------------------
if __name__ == "__main__":

    utils = Utils()
    utils.download_datasets()

    study = optuna.create_study(
        direction="maximize",
        study_name="MSML_AUCROC_ADBench_FULL"
    )

    study.optimize(
        objective,
        n_trials=10,
        callbacks=[print_trial_result]
    )

    print(" Best AUCROC:", study.best_value)
    print(" Best hyperparameters:", study.best_params)

    df = study.trials_dataframe()
    df.to_csv(
        "adbench/result/MSDE_optuna_dependency_none_noise.csv",
        index=False
    )

    print(" Saved to adbench/result/MSDE_optuna_dependency_none_noise.csv")


if there is any question while downloading datasets, we suggest you to download it from the website:
https://github.com/Minqi824/ADBench/tree/main/adbench/datasets
如果您在中国大陆地区，请使用链接：
https://jihulab.com/BraudoCC/ADBench_datasets/
100% [................................................................................] 3852 / 3852

100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 421.31it/s]
[I 2026-01-07 10:39:27,329] A new study created in memory with name: MSML_AUCROC_ADBench_FULL


CIFAR10_0.npz already exists. Skipping download...
CIFAR10_1.npz already exists. Skipping download...
CIFAR10_2.npz already exists. Skipping download...
CIFAR10_3.npz already exists. Skipping download...
CIFAR10_4.npz already exists. Skipping download...
CIFAR10_5.npz already exists. Skipping download...
CIFAR10_6.npz already exists. Skipping download...
CIFAR10_7.npz already exists. Skipping download...
CIFAR10_8.npz already exists. Skipping download...
CIFAR10_9.npz already exists. Skipping download...
FashionMNIST_0.npz already exists. Skipping download...
FashionMNIST_1.npz already exists. Skipping download...
FashionMNIST_2.npz already exists. Skipping download...
FashionMNIST_3.npz already exists. Skipping download...
FashionMNIST_4.npz already exists. Skipping download...
FashionMNIST_5.npz already exists. Skipping download...
FashionMNIST_6.npz already exists. Skipping download...
FashionMNIST_7.npz already exists. Skipping download...
FashionMNIST_8.npz already exists. Skippin

0it [00:00, ?it/s]

generating duplicate samples for dataset 15_Hepatitis...
current noise type: None
{'Samples': 1000, 'Features': 19, 'Anomalies': np.int64(170), 'Anomalies Ratio(%)': np.float64(17.0)}


...

Error when generating data: Constant column.
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 50, 'Anomalies': np.int64(628), 'Anomalies Ratio(%)': np.float64(6.28)}
Model: Customized, AUC-ROC: 0.9506560031476045, AUC-PR: 0.46877779861694


1082it [45:17,  4.17s/it]

Current experiment parameters: ('9_census', 0.0, np.int64(2)), model: Customized, metrics: {'aucroc': np.float64(0.9506560031476045), 'aucpr': np.float64(0.46877779861694), 'p_at_n': np.float64(0.4521276595744681), 'adj_p_at_n': np.float64(0.41549892557731305), 'adj_ap': np.float64(0.4332622318103912)}, fitting time: 1.430511474609375e-06, inference time: 3.75530743598938
subsampling for dataset 9_census...
Generating dependency anomalies...


C:\Users\user\anaconda3\Lib\site-packages\copulas\multivariate\vine.py:78: UserWarning: Vines have not been fully tested on Python >= 3.8 and might produce wrong results.
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\copulas\bivariate\base.py:163: RuntimeWarning: Data does not appear to be uniform.
  warnings.warn('Data does not appear to be uniform.', category=RuntimeWarning)
1104it [46:47,  2.54s/it]
[I 2026-01-07 18:42:54,178] Trial 9 finished with value: 0.8375470653862023 and parameters: {'k': 99, 'nbd_sample_count_threshold': 22, 'learning_rate': 0.0744462688592306, 'max_iters_shift': 6, 'shift_threshold': 0.00026294560468682036, 'anomalyThreshold': 0.1756872729485472}. Best is trial 4 with value: 0.8798693755896347.


Error when generating data: Constant column.

================ Trial Finished ================
Trial number : 9
AUCROC       : 0.8375470653862023
Hyperparameters:
  k: 99
  nbd_sample_count_threshold: 22
  learning_rate: 0.0744462688592306
  max_iters_shift: 6
  shift_threshold: 0.00026294560468682036
  anomalyThreshold: 0.1756872729485472

 Best AUCROC: 0.8798693755896347
 Best hyperparameters: {'k': 13, 'nbd_sample_count_threshold': 35, 'learning_rate': 0.09450702450316578, 'max_iters_shift': 19, 'shift_threshold': 0.00022482110799483155, 'anomalyThreshold': 0.29426545216477157}
 Saved to adbench/result/MSDE_optuna_dependency_none_noise.csv
